In [1]:
import numpy as np
import pandas as pd

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

In [3]:
sales_data = pd.read_excel('C:/Users/Henil/Downloads/Sales Dump Report_export.xlsx',parse_dates=["Date Of Invoice"])  
return_data = pd.read_excel('C:/Users/Henil/Downloads/Sales Return Report_export.xlsx',parse_dates=["SR Document Date"])
inventory_data = pd.read_excel('C:/Users/Henil/Downloads/Stock Report_export_.xlsx',parse_dates=["Expiry Date"])

C:\Users\Henil\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [4]:
sales_df = sales_data.rename(columns={
    "Store Code": "store_code",
    "Date Of Invoice": "date",
    "Product Code": "product_code",
    "BatchNo": "batch_no",
    "Qty": "qty"
})

returns_df = return_data.rename(columns={
    "Store Code": "store_code",
    "SR Document Date": "date",
    "Product Code": "product_code",
    "BatchNo": "batch_no",
    "Qty": "qty"
})

closing_df = inventory_data.rename(columns={
    "Store Code": "store_code",
    "Product Code": "product_code",
    "Batch No": "batch_no",
    "Qty": "closing_qty",
    "Expiry Date": "expiry_date",
    "Sales Rate": "sales_rate",
    "Purchase Rate": "purchase_rate",
    "MRP": "mrp"
})


## Closing Inventory Table

In [5]:
inventory_snapshot = closing_df[
    [
        "store_code",
        "product_code",
        "batch_no",
        "closing_qty",
        "expiry_date",
        "sales_rate",
        "purchase_rate",
        "mrp"
    ]
].copy()


# Phase 1

## Sales & Returns Movement Ledger

In [6]:
sales_movement = sales_df[
    ["store_code", "product_code", "batch_no", "date", "qty"]
].copy()

sales_movement["movement_qty"] = -sales_movement["qty"]
sales_movement["movement_type"] = "SALE"

In [7]:
returns_movement = returns_df[
    ["store_code", "product_code","batch_no", "date", "qty"]
].copy()

returns_movement["movement_qty"] = returns_movement["qty"]
returns_movement["movement_type"] = "RETURN"


In [8]:
movement_ledger = pd.concat([sales_movement, returns_movement], ignore_index=True)

movement_ledger = movement_ledger[
    [
        "store_code",
        "product_code",
        "batch_no",
        "date",
        "movement_qty",
        "movement_type"
    ]
]


In [9]:
today = pd.Timestamp.today().normalize()

def net_sales(days):
    return (
        movement_ledger[movement_ledger["date"] >= today - pd.Timedelta(days=days)]
        .groupby(["store_code", "product_code", "batch_no"])["movement_qty"]
        .sum()
        .reset_index(name=f"net_sales_{days}d")
    )

net_30 = net_sales(30)
net_60 = net_sales(60)
net_90 = net_sales(90)


In [10]:
last_sale = (
    movement_ledger[movement_ledger["movement_type"] == "SALE"]
    .groupby(["store_code", "product_code", "batch_no"])["date"]
    .max()
    .reset_index(name="last_sale_date")
)

activity_table = inventory_snapshot.merge(last_sale, how="left",
                                           on=["store_code", "product_code", "batch_no"])

activity_table["days_since_last_sale"] = (
    today - activity_table["last_sale_date"]
).dt.days


In [11]:
# add Period For Non Moving Stocks "Currently 60 Days"

In [12]:
activity_table["non_moving_flag"] = activity_table["days_since_last_sale"] >= 60

In [13]:
# Assume net_30, net_60, net_90 exist from Phase 1

activity_table = activity_table.merge(
    net_30, 
    on=["store_code", "product_code", "batch_no"], 
    how="left"
)
activity_table = activity_table.merge(
    net_60, 
    on=["store_code", "product_code", "batch_no"], 
    how="left"
)
activity_table = activity_table.merge(
    net_90, 
    on=["store_code", "product_code", "batch_no"], 
    how="left"
)

# Fill NaN with 0 (for products with no sales in that period)
activity_table[["net_sales_30d", "net_sales_60d", "net_sales_90d"]] = activity_table[
    ["net_sales_30d", "net_sales_60d", "net_sales_90d"]
].fillna(0)


### Calculating Expiry

In [14]:
activity_table["days_to_expiry"] = (
    activity_table["expiry_date"] - today
).dt.days


In [15]:
def expiry_bucket(days):
    if days < 0:
        return "Expired ☠"
    elif days <= 30:
        return "Expiring in 0–30 days 🔴"
    elif days <= 60:
        return "Expiring in 30–60 days 🟠"
    elif days <= 90:
        return "Expiring in 60–90 days 🟡"
    else:
        return "Safe 🟢"

activity_table["expiry_status"] = activity_table["days_to_expiry"].apply(expiry_bucket)


## Phase-1 Output DF

In [16]:
# inventory_snapshot : What stock is available now
# movement_ledger : What moved in/out and when
# activity_table : 
    #Ready for:
        #Non-moving
        #Dead stock
        #Expiry risk
        #Slow / Fast moving
        #Risk engine
        #Forecasting

In [17]:
activity_table.head(1)

,store_code,product_code,batch_no,closing_qty,expiry_date,sales_rate,purchase_rate,mrp,last_sale_date,days_since_last_sale,non_moving_flag,net_sales_30d,net_sales_60d,net_sales_90d,days_to_expiry,expiry_status
0,1078,2531,Di/2380,593,2026-09-30 00:00:10,0.25,0.25,40.00,NaT,NaN,False,0.00,0.00,0.00,166,Safe 🟢


# Phase - 2

In [18]:
# Non Moving
non_moving_threshold = 60
activity_table["non_moving_flag"] = activity_table["days_since_last_sale"] >= non_moving_threshold


In [19]:
activity_table.columns

Index(['store_code', 'product_code', 'batch_no', 'closing_qty', 'expiry_date',
       'sales_rate', 'purchase_rate', 'mrp', 'last_sale_date',
       'days_since_last_sale', 'non_moving_flag', 'net_sales_30d',
       'net_sales_60d', 'net_sales_90d', 'days_to_expiry', 'expiry_status'],
      dtype='object')

In [20]:
# Velocity = net_sales_30d / closing_qty (if closing_qty > 0)
# for change net_sale_30d to 60d and 90d
activity_table["velocity_30d"] = np.where(
    activity_table["net_sales_30d"] < 0,
    abs(activity_table["net_sales_30d"]) /
    activity_table["closing_qty"].replace(0, np.nan),
    0
)

slow_moving_threshold = 0.1  # customizable
activity_table["slow_moving_flag"] = (
    (activity_table["net_sales_30d"] < 0) &
    (activity_table["velocity_30d"] <= slow_moving_threshold)
)


activity_table["return_heavy_flag"] = activity_table["net_sales_30d"] > 0

In [21]:
activity_table["return_ratio"] = (
    activity_table["net_sales_30d"].clip(lower=0) /  # only positive values
    activity_table["closing_qty"].replace(0, np.nan)
)


activity_table["high_return_flag"] = activity_table["return_ratio"] > 0.05


In [22]:
activity_table["fast_moving_flag"] = (
    (activity_table["net_sales_30d"] < 0) &
    (activity_table["velocity_30d"] > slow_moving_threshold)
)


In [23]:
activity_table["non_moving_flag"] = activity_table["net_sales_60d"] == 0

In [24]:
activity_table["dead_stock_flag"] = (
    activity_table["non_moving_flag"] &
    (activity_table["closing_qty"] > 0)
)


In [25]:
def classify_stock(row):
    if row["dead_stock_flag"]:
        return "Dead Stock ☠"
    elif row["non_moving_flag"]:
        return "Non-Moving 💀"
    elif row["slow_moving_flag"]:
        return "Slow Moving 🐢"
    elif row["fast_moving_flag"]:
        return "Fast Moving ✅"
    else:
        return "Unclassified"

activity_table["stock_classification"] = activity_table.apply(classify_stock, axis=1)


In [26]:
# prioritize high-value dead or slow-moving stock.
activity_table["stock_value"] = activity_table["closing_qty"] * activity_table["sales_rate"]


In [28]:
activity_table.head(2)

,store_code,product_code,batch_no,closing_qty,expiry_date,sales_rate,purchase_rate,mrp,last_sale_date,days_since_last_sale,non_moving_flag,net_sales_30d,net_sales_60d,net_sales_90d,days_to_expiry,expiry_status,velocity_30d,slow_moving_flag,return_heavy_flag,return_ratio,high_return_flag,fast_moving_flag,dead_stock_flag,stock_classification,stock_value
0,1078,2531,Di/2380,593,2026-09-30 00:00:10,0.25,0.25,40.00,NaT,NaN,True,0.00,0.00,0.00,166,Safe 🟢,0.00,False,False,0.00,False,False,True,Dead Stock ☠,148.25
1,1078,2531,Di/2379,164,2026-09-30 00:00:10,0.25,0.25,40.00,2024-09-21 00:00:10,572.00,True,0.00,0.00,0.00,166,Safe 🟢,0.00,False,False,0.00,False,False,True,Dead Stock ☠,41.00


## stock_classification_table

| Column               | Description                     |
| -------------------- | ------------------------------- |
| store_code           | Store identifier                |
| product_code         | Product identifier              |
| batch_no             | Batch identifier                |
| closing_qty          | Current stock                   |
| days_since_last_sale | Days since last sale            |
| velocity_30d         | Recent velocity                 |
| stock_classification | Fast / Slow / Non-Moving / Dead |
| days_to_expiry       | Days until expiry               |
| expiry_status        | Expiry bucket (from Phase 1)    |
| stock_value          | Stock value for prioritization  |


# Phase 3

Risk Point Distribution 
Near Expiry: 40 points
Low Stock: 30 points
Overstock: 20 points
Dead Stock / Non-Moving: 50 points
High Return Ratio: 25 points


| Risk Type         | Logic                                            | Points |
| ----------------- | ------------------------------------------------ | ------ |
| Expired           | days_to_expiry < 0                               | 50     |
| Near Expiry       | 0–60 days                                        | 40     |
| Mid Near Expiry   | 61–90 days                                       | 30     |
| Long Near Expiry  | 91–180 days                                      | 20     |
| Low Stock         | closing_qty < min_threshold                      | 30     |
| Overstock         | closing_qty > forecast × 1.5                     | 20     |
| Dead / Non-Moving | stock_classification in [Dead Stock, Non-Moving] | 50     |
| High Return Ratio | return_ratio > 0.2                               | 25     |


In [29]:
def expiry_bucket_days(days):
    if days < 0:
        return "Expired"
    elif days <= 60:
        return "0-60 days"
    elif days <= 90:
        return "61-90 days"
    elif days <= 180:
        return "91-180 days"
    else:
        return ">180 days"

activity_table["expiry_bucket_days"] = activity_table["days_to_expiry"].apply(expiry_bucket_days)


In [30]:
def calculate_risk(row):
    risk_score = 0
    risk_reasons = []

    # Expiry risk
    if row["expiry_bucket_days"] == "Expired":
        risk_score += 50
        risk_reasons.append("Expired")
    elif row["expiry_bucket_days"] == "0-60 days":
        risk_score += 40
        risk_reasons.append("Near Expiry 0-60")
    elif row["expiry_bucket_days"] == "61-90 days":
        risk_score += 30
        risk_reasons.append("Near Expiry 61-90")
    elif row["expiry_bucket_days"] == "91-180 days":
        risk_score += 20
        risk_reasons.append("Near Expiry 91-180")

    # Low stock
    min_threshold = row.get("min_threshold", 10)  # default safety stock
    if row["closing_qty"] < min_threshold:
        risk_score += 30
        risk_reasons.append("Low Stock")

    # Dead / Non-Moving
    if row["stock_classification"] in ["Dead Stock ☠", "Non-Moving 💀"]:
        risk_score += 50
        risk_reasons.append("Dead / Non-Moving")

    # Optional: High return ratio
    if "return_ratio" in row and row["return_ratio"] > 0.2:
        risk_score += 25
        risk_reasons.append("High Return Ratio")

    # Map score to status
    if risk_score >= 70:
        status = "High 🔴"
    elif risk_score >= 40:
        status = "Medium 🟠"
    else:
        status = "Low 🟢"

    return pd.Series([risk_score, status, ", ".join(risk_reasons)])


In [31]:
activity_table[["risk_score", "risk_status", "risk_reasons"]] = activity_table.apply(calculate_risk, axis=1)


In [32]:
expiry_counts_store = activity_table.groupby(
    ["store_code", "product_code", "expiry_bucket_days"]
)["closing_qty"].sum().reset_index(name="qty_in_bucket")

expiry_pivot = expiry_counts_store.pivot_table(
    index=["store_code", "product_code"],
    columns="expiry_bucket_days",
    values="qty_in_bucket",
    fill_value=0
).reset_index()


activity_table enhanced with:

risk_score → 0–100

risk_status → High / Medium / Low

risk_reasons → Which risks contributed

expiry_bucket_days → 0–60 / 61–90 / 91–180 / >180 / Expired

expiry_pivot → store/product-level expiry quantity dashboard

# Phase 4

In [33]:
# activity_table["forecast_30d"] = (
#     0.5*activity_table["net_sales_30d"] +
#     0.3*activity_table["net_sales_60d"] +
#     0.2*activity_table["net_sales_90d"]
# )

sales_0_30   = activity_table["net_sales_30d"]
sales_31_60  = activity_table["net_sales_60d"] - activity_table["net_sales_30d"]
sales_61_90  = activity_table["net_sales_90d"] - activity_table["net_sales_60d"]

activity_table["forecast_30d"] = (
    0.5*sales_0_30 +
    0.3*sales_31_60 +
    0.2*sales_61_90
)

activity_table.loc[
    activity_table["forecast_30d"] > 0, "forecast_30d"
] = 0


In [34]:
# # Aggregate batch-wise forecast to product-level
# product_forecast = activity_table.groupby("product_name")["forecast_30d"].sum().reset_index()

# # Rename the column for clarity
# product_forecast = product_forecast.rename(columns={"forecast_30d": "product_forecast_30d"})

# # Merge back into activity_table
# activity_table = activity_table.merge(product_forecast, on="product_name", how="left")


In [35]:
product_store_forecast = (
    activity_table.groupby(["store_code", "product_code"])["forecast_30d"]
    .sum()
    .reset_index()
    .rename(columns={"forecast_30d": "product_forecast_30d"})
)

# Merge back to activity_table
activity_table = activity_table.merge(product_store_forecast, on=["store_code", "product_code"], how="left")


In [36]:
activity_table.head(2)


,store_code,product_code,batch_no,closing_qty,expiry_date,sales_rate,purchase_rate,mrp,last_sale_date,days_since_last_sale,non_moving_flag,net_sales_30d,net_sales_60d,net_sales_90d,days_to_expiry,expiry_status,velocity_30d,slow_moving_flag,return_heavy_flag,return_ratio,high_return_flag,fast_moving_flag,dead_stock_flag,stock_classification,stock_value,expiry_bucket_days,risk_score,risk_status,risk_reasons,forecast_30d,product_forecast_30d
0,1078,2531,Di/2380,593,2026-09-30 00:00:10,0.25,0.25,40.00,NaT,NaN,True,0.00,0.00,0.00,166,Safe 🟢,0.00,False,False,0.00,False,False,True,Dead Stock ☠,148.25,91-180 days,70,High 🔴,"Near Expiry 91-180, Dead / Non-Moving",0.00,0.00
1,1078,2531,Di/2379,164,2026-09-30 00:00:10,0.25,0.25,40.00,2024-09-21 00:00:10,572.00,True,0.00,0.00,0.00,166,Safe 🟢,0.00,False,False,0.00,False,False,True,Dead Stock ☠,41.00,91-180 days,70,High 🔴,"Near Expiry 91-180, Dead / Non-Moving",0.00,0.00


In [37]:
#Reorder_Qty = Forecasted_Demand + Safety_Stock - Closing_Stock + Expiry_Adjustment

In [38]:
# Safety stock default
if "safety_stock" not in activity_table.columns:
    activity_table["safety_stock"] = 10
else:
    activity_table["safety_stock"] = activity_table["safety_stock"].fillna(10)

# Expiry adjustment: reduce recommended order if batch near expiry
activity_table["expiry_adjustment"] = activity_table["closing_qty"].where(
    activity_table["days_to_expiry"] <= 60, 0
)

# Usable stock
activity_table["usable_stock"] = (
    activity_table["closing_qty"] - activity_table["expiry_adjustment"]
)




In [39]:
activity_table["reorder_qty_batch"] = (
    abs(activity_table["forecast_30d"]) +
    activity_table["safety_stock"] -
    activity_table["usable_stock"]
).clip(lower=0)


In [40]:
activity_table["reorder_qty_product"] = (
    abs(activity_table["product_forecast_30d"]) +
    activity_table.groupby(["store_code", "product_code"])["safety_stock"].transform("sum") -
    activity_table.groupby(["store_code", "product_code"])["usable_stock"].transform("sum")
).clip(lower=0)


In [41]:
def generate_alert(row):
    alerts = []
    if row["reorder_qty_product"] > 0 and row["risk_status"] == "High 🔴":
        alerts.append("URGENT REPLENISHMENT")
    elif row["reorder_qty_product"] > 0:
        alerts.append("Replenish Stock")
    
    if row["days_to_expiry"] <= 30:
        alerts.append("Near Expiry Clearance")
    
    if row["stock_classification"] in ["Dead Stock ☠", "Non-Moving 💀"]:
        alerts.append("Consider Clearance / Promotion")
    
    return ", ".join(alerts) if alerts else "No Action"

activity_table["alerts"] = activity_table.apply(generate_alert, axis=1)


In [42]:
# Keep only rows that have any alerts
alerts_table = activity_table[activity_table["alerts"] != "No Action"].copy()


In [43]:
alerts_table = alerts_table[[
    "store_code",
    "product_code",
    "batch_no",
    "closing_qty",
    "days_to_expiry",
    "expiry_bucket_days",
    "stock_classification",
    "risk_score",
    "risk_status",   
    "alerts",
    "reorder_qty_batch",
    "reorder_qty_product"   
]]


In [44]:
alerts_table = alerts_table.sort_values(by=["risk_score", "days_to_expiry"], ascending=[False, True])

# Final DF

activity_table → The master table

Batch-level, store-level, product-level data

Stock classification, risk scores, days-to-expiry, forecast, reorder quantities, alerts

In [45]:
activity_table.head(2)

,store_code,product_code,batch_no,closing_qty,expiry_date,sales_rate,purchase_rate,mrp,last_sale_date,days_since_last_sale,non_moving_flag,net_sales_30d,net_sales_60d,net_sales_90d,days_to_expiry,expiry_status,velocity_30d,slow_moving_flag,return_heavy_flag,return_ratio,high_return_flag,fast_moving_flag,dead_stock_flag,stock_classification,stock_value,expiry_bucket_days,risk_score,risk_status,risk_reasons,forecast_30d,product_forecast_30d,safety_stock,expiry_adjustment,usable_stock,reorder_qty_batch,reorder_qty_product,alerts
0,1078,2531,Di/2380,593,2026-09-30 00:00:10,0.25,0.25,40.00,NaT,NaN,True,0.00,0.00,0.00,166,Safe 🟢,0.00,False,False,0.00,False,False,True,Dead Stock ☠,148.25,91-180 days,70,High 🔴,"Near Expiry 91-180, Dead / Non-Moving",0.00,0.00,10,0,593,0.00,0.00,Consider Clearance / Promotion
1,1078,2531,Di/2379,164,2026-09-30 00:00:10,0.25,0.25,40.00,2024-09-21 00:00:10,572.00,True,0.00,0.00,0.00,166,Safe 🟢,0.00,False,False,0.00,False,False,True,Dead Stock ☠,41.00,91-180 days,70,High 🔴,"Near Expiry 91-180, Dead / Non-Moving",0.00,0.00,10,0,164,0.00,0.00,Consider Clearance / Promotion


expiry_pivot → Dashboard / reporting table for expiry

Aggregates stock by expiry buckets (0–60, 61–90, 91–180, >180, Expired)

Helps visualize near-expiry stock per store/product

In [46]:
expiry_pivot = expiry_pivot.merge(
    activity_table['store_code'].drop_duplicates(),
    on='store_code',
    how='left'
)


In [47]:
expiry_pivot.head(2)

,store_code,product_code,0-60 days,61-90 days,91-180 days,>180 days,Expired
0,593,6972,0.00,0.00,0.00,150.00,0.00
1,593,6973,0.00,0.00,0.00,160.00,0.00


alerts_table → Filtered action table

Contains only rows that require urgent attention (replenishment, clearance, near-expiry, dead stock)

Useful for sending alerts, reports, or triggering management actions

In [49]:
alerts_table.head(2)

,store_code,product_code,batch_no,closing_qty,days_to_expiry,expiry_bucket_days,stock_classification,risk_score,risk_status,alerts,reorder_qty_batch,reorder_qty_product
1041,593,7992,UGT24034A,9,-107,Expired,Dead Stock ☠,130,High 🔴,"Near Expiry Clearance, Consider Clearance / Pr...",10.00,0.00
1170,1080,8501,AGC230256A,6,-107,Expired,Dead Stock ☠,130,High 🔴,"URGENT REPLENISHMENT, Near Expiry Clearance, C...",10.00,10.00


# Check Order Details

In [50]:
# Assuming `order_products` is a list of product_codes from the order
store_code = "1080"
order_products = [8501]  # example product codes

order_health = activity_table[
    (activity_table["store_code"] == store_code) &
    (activity_table["product_code"].isin(order_products))
].copy()


In [51]:
order_health = order_health[[
    "store_code",
    "product_code",
    "batch_no",
    "closing_qty",
    "stock_classification",
    "risk_score",
    "risk_status",
    "forecast_30d",
    "days_to_expiry",
    "alerts"
]]


In [52]:
import pandas as pd

# Suppose you have these DataFrames
# activity_table, expiry_pivot, alerts_table

# Path to save Excel
file_path = r"C:\Users\Henil\Downloads\Inventory.xlsx"

# Write to Excel with multiple sheets
with pd.ExcelWriter(file_path, engine='xlsxwriter') as writer:
    activity_table.to_excel(writer, sheet_name="Batchwise Stock", index=False)
    expiry_pivot.to_excel(writer, sheet_name="Expiry Stock Breakdown", index=False)
    alerts_table.to_excel(writer, sheet_name="Stock Alert", index=False)


